In [18]:
import mujoco
mjcf_path = "../models/ur10/ur10.xml"
urdf_path = "../models/ur10/ur10.urdf"
mj_model = mujoco.MjModel.from_xml_path(urdf_path)

# 2. Zapisujemy go jako natywny plik MuJoCo (MJCF)
mujoco.mj_saveLastXML(mjcf_path, mj_model)

In [2]:
from optimal.PDController import PDController
import pinocchio as pin
import numpy as np
import time
import mujoco
import mujoco.viewer

mjcf_path = "../models/ur10/ur10.xml"
urdf_path = "../models/ur10/ur10.urdf"
robot_mj_model = mujoco.MjModel.from_xml_path(mjcf_path)
robot_pin_model = pin.buildModelFromUrdf(urdf_path)

kp = np.array([400.0, 800.0, 400.0, 100.0, 40.0, 10.0])
kd = np.array([40.0,  80.0,  40.0,  10.0,  4.0,  1.0])
torque_limits = np.array([330.0, 330.0, 150.0, 56.0, 56.0, 56.0])
controller = PDController(robot_pin_model, kp=kp, kd=kd, torque_limits=torque_limits)

DT = robot_mj_model.opt.timestep
q_start = pin.neutral(robot_pin_model)
q_target = np.array([0.7, -1.2, 1.5, -0.8, 1.5, 0.5])
x_start = np.concatenate([q_start, np.zeros(robot_pin_model.nv)])
x_target = np.concatenate([q_target, np.zeros(robot_pin_model.nv)])
xs, us = controller.compute_control(x_start=x_start, x_goal=x_target, horizon=1500, control_dt=DT)

robot_mj_data = mujoco.MjData(robot_mj_model)
if True:
    with mujoco.viewer.launch_passive(robot_mj_model, robot_mj_data) as viewer:
        for i, u_cmd in enumerate(us):
            # print(u_cmd)
            # robot_mj_data.ctrl[:] = u_cmd
            # robot_mj_data.qpos[:] = q_target
            robot_mj_data.qpos[:] = xs[i][: robot_pin_model.nq]
            mujoco.mj_forward(robot_mj_model, robot_mj_data)
            # mujoco.mj_step(robot_mj_model, robot_mj_data)
            viewer.sync()
            time.sleep(DT)
        input("Press Enter to continue...")
        


In [ ]:
from optimal.PDController import PDController
import pinocchio as pin
import numpy as np
import time
import mujoco
import mujoco.viewer

mjcf_path = "../models/ur10/ur10.xml"
urdf_path = "../models/ur10/ur10.urdf"
robot_mj_model = mujoco.MjModel.from_xml_path(mjcf_path)
robot_pin_model = pin.buildModelFromUrdf(urdf_path)

kp = np.array([400.0, 800.0, 400.0, 100.0, 40.0, 10.0])
kd = np.array([40.0,  80.0,  40.0,  10.0,  4.0,  1.0])
torque_limits = np.array([330.0, 330.0, 150.0, 56.0, 56.0, 56.0])
controller = PDController(robot_pin_model, kp=kp, kd=kd, torque_limits=torque_limits)

DT = robot_mj_model.opt.timestep
q_start = pin.neutral(robot_pin_model)
q_target = np.array([0.7, -1.2, 1.5, -0.8, 1.5, 0.5])
x_start = np.concatenate([q_start, np.zeros(robot_pin_model.nv)])
x_target = np.concatenate([q_target, np.zeros(robot_pin_model.nv)])
xs, us = controller.compute_control(x_start=x_start, x_goal=x_target, horizon=1500, control_dt=DT, alpha=0.8)

robot_mj_data = mujoco.MjData(robot_mj_model)
if True:
    with mujoco.viewer.launch_passive(robot_mj_model, robot_mj_data) as viewer:
        for i, u_cmd in enumerate(us):
            # print(u_cmd)
            # robot_mj_data.ctrl[:] = u_cmd
            # robot_mj_data.qpos[:] = q_target
            robot_mj_data.qpos[:] = xs[i][: robot_pin_model.nq]
            mujoco.mj_forward(robot_mj_model, robot_mj_data)
            # mujoco.mj_step(robot_mj_model, robot_mj_data)
            viewer.sync()
            time.sleep(DT)
        input("Press Enter to continue...")
        


In [11]:
import crocoddyl
import pinocchio
import numpy as np

class DifferentialFreeFwdDynamicsModelDerived(
    crocoddyl.DifferentialActionModelAbstract
):
    def __init__(self, state, actuationModel, costModel):
        crocoddyl.DifferentialActionModelAbstract.__init__(
            self, state, actuationModel.nu, costModel.nr
        )
        self.actuation = actuationModel
        self.costs = costModel
        self.enable_force = True
        self.armature = np.matrix(np.zeros(0))

    def calc(self, data, x, u=None):
        if u is None:
            q, v = x[: self.state.nq], x[-self.state.nv :]
            pinocchio.computeAllTerms(self.state.pinocchio, data.pinocchio, q, v)
            self.costs.calc(data.costs, x)
            data.cost = data.costs.cost
        else:
            q, v = x[: self.state.nq], x[-self.state.nv :]
            self.actuation.calc(data.actuation, x, u)
            tau = data.actuation.tau
            # Computing the dynamics using ABA or manually for armature case
            if self.enable_force:
                data.xout[:] = pinocchio.aba(
                    self.state.pinocchio, data.pinocchio, q, v, tau
                )
            else:
                pinocchio.computeAllTerms(self.state.pinocchio, data.pinocchio, q, v)
                data.M = data.pinocchio.M
                if self.armature.size == self.state.nv:
                    data.M[range(self.state.nv), range(self.state.nv)] += self.armature
                data.Minv = np.linalg.inv(data.M)
                data.xout[:] = np.dot(data.Minv, (tau - data.pinocchio.nle))
            # Computing the cost value and residuals
            pinocchio.forwardKinematics(self.state.pinocchio, data.pinocchio, q, v)
            pinocchio.updateFramePlacements(self.state.pinocchio, data.pinocchio)
            self.costs.calc(data.costs, x, u)
            data.cost = data.costs.cost

    def calcDiff(self, data, x, u=None):
        if u is None:
            self.costs.calcDiff(data.costs, x)
        else:
            nq, nv = self.state.nq, self.state.nv
            q, v = x[:nq], x[-nv:]
            # Computing the actuation derivatives
            self.actuation.calcDiff(data.actuation, x, u)
            tau = data.actuation.tau
            # Computing the dynamics derivatives
            if self.enable_force:
                pinocchio.computeABADerivatives(
                    self.state.pinocchio, data.pinocchio, q, v, tau
                )
                ddq_dq = data.pinocchio.ddq_dq
                ddq_dv = data.pinocchio.ddq_dv
                data.Fx[:, :] = np.hstack([ddq_dq, ddq_dv]) + np.dot(
                    data.pinocchio.Minv, data.actuation.dtau_dx
                )
                data.Fu[:, :] = np.dot(data.pinocchio.Minv, data.actuation.dtau_du)
            else:
                pinocchio.computeRNEADerivatives(
                    self.state.pinocchio, data.pinocchio, q, v, data.xout
                )
                ddq_dq = np.dot(
                    data.Minv, (data.actuation.dtau_dx[:, :nv] - data.pinocchio.dtau_dq)
                )
                ddq_dv = np.dot(
                    data.Minv, (data.actuation.dtau_dx[:, nv:] - data.pinocchio.dtau_dv)
                )
                data.Fx[:, :] = np.hstack([ddq_dq, ddq_dv])
                data.Fu[:, :] = np.dot(data.Minv, data.actuation.dtau_du)
            # Computing the cost derivatives
            self.costs.calcDiff(data.costs, x, u)

    def createData(self):
        data = DifferentialFreeFwdDynamicsDataDerived(self)
        return data

    def set_armature(self, armature):
        if armature.size is not self.state.nv:
            print("The armature dimension is wrong, we cannot set it.")
        else:
            self.enable_force = False
            self.armature = armature.T


class DifferentialFreeFwdDynamicsDataDerived(crocoddyl.DifferentialActionDataAbstract):
    def __init__(self, model):
        crocoddyl.DifferentialActionDataAbstract.__init__(self, model)
        self.pinocchio = pinocchio.Model.createData(model.state.pinocchio)
        self.multibody = crocoddyl.DataCollectorMultibody(self.pinocchio)
        self.actuation = model.actuation.createData()
        self.costs = model.costs.createData(self.multibody)
        self.costs.shareMemory(self)
        self.Minv = None

In [1]:
import time

import numpy as np
import mujoco
import mujoco.viewer
import pinocchio as pin
import optimal.PDController

mjcf_path = "../models/ur10/ur10.xml"
urdf_path = "../models/ur10/ur10.urdf"
robot_mj_model = mujoco.MjModel.from_xml_path(mjcf_path)
robot_pin_model = pin.buildModelFromUrdf(urdf_path)
robot_pin_model.gravity = pin.Motion(np.array([0, 0, -9.81, 0, 0, 0]))
# print(type(robot_pin_model))
# print(robot_pin_model.nq)
# print(robot_pin_model.nv)
# q_ref = pin.neutral(robot_pin_model)
# reduced_model = pin.buildReducedModel(robot_pin_model, [1], q_ref)
# print(type(reduced_model))
# print(robot_pin_model.nq)
# print(robot_pin_model.nv)

q0 = pin.neutral(robot_pin_model)
x0 = np.concatenate([q0, np.zeros(robot_pin_model.nv)])

q_target = np.array([0.0, -1.57, 2.2, -1.761211, -0.306910, 1.047198])
x_target = np.concatenate([q_target, np.zeros(robot_pin_model.nv)])


controller = optimal.PDController.PDController(robot_pin_model, kp=3.0, kd=0.2, torque_limits=robot_mj_model.actuator_ctrlrange[:, 1])
xs, us = controller.compute_control(x0, x_target, horizon=3000, control_dt=robot_mj_model.opt.timestep)

robot_mj_data = mujoco.MjData(robot_mj_model)
if True:
    with mujoco.viewer.launch_passive(robot_mj_model, robot_mj_data) as viewer:
        for i, u_cmd in enumerate(us):
            print(u_cmd)
            # robot_mj_data.ctrl[:] = u_cmd
            # robot_mj_data.qpos[:] = q_target
            robot_mj_data.qpos[:] = xs[i][: robot_pin_model.nq]
            mujoco.mj_forward(robot_mj_model, robot_mj_data)
            # mujoco.mj_step(robot_mj_model, robot_mj_data)
            viewer.sync()
            time.sleep(robot_mj_model.opt.timestep)
        input("Press Enter to continue...")
        

[ 0.       -4.71      6.6      -5.283633 -0.92073   3.141594]
[-3.68595282e-04 -4.71292701e+00  6.59671633e+00 -5.12034811e+00
 -6.35753586e-01 -8.22352934e-01]
[-5.68166723e-04 -4.71597484e+00  6.59343498e+00 -5.02572491e+00
 -4.63310895e-01 -2.50260430e-02]
[-8.24543047e-04 -4.71912947e+00  6.59011073e+00 -4.92072494e+00
 -2.90508938e-01 -1.75063018e-01]
[-1.09558467e-03 -4.72239355e+00  6.58674490e+00 -4.82072913e+00
 -1.38089913e-01 -1.37221825e-01]
[-1.39055518e-03 -4.72576636e+00  6.58332852e+00 -4.72237471e+00
 -3.49082641e-04 -1.37199775e-01]
[-1.70701879e-03 -4.72924795e+00  6.57985440e+00 -4.62605507e+00
  1.23041203e-01 -1.30071634e-01]
[-2.04389155e-03 -4.73283825e+00  6.57631471e+00 -4.53144102e+00
  2.33369663e-01 -1.24734129e-01]
[-2.39913052e-03 -4.73653724e+00  6.57270167e+00 -4.43836768e+00
  3.31635169e-01 -1.19387687e-01]
[-2.77045124e-03 -4.74034497e+00  6.56900756e+00 -4.34665482e+00
  4.18803686e-01 -1.14365410e-01]
[-3.15537681e-03 -4.74426149e+00  6.56522483e+0

In [5]:
import time
import numpy as np
import pinocchio as pin
from pinocchio.visualize import MeshcatVisualizer

# We use example_robot_data to get a perfect UR10 model out-of-the-box
import example_robot_data

def main():
    print("Loading UR10 robot model...")
    # 1. Load the robot (this includes the URDF, meshes, and kinematics)
    robot = example_robot_data.load('ur10')

    # 2. Initialize the Meshcat Visualizer
    # We pass the kinematic model, collision geometries, and visual geometries
    viz = MeshcatVisualizer(robot.model, robot.collision_model, robot.visual_model)

    # Start the Meshcat server and open it in the default web browser
    print("Starting Meshcat server. Look for a new tab in your browser!")
    viz.initViewer(open=True)
    
    # Load the 3D meshes into the viewer
    viz.loadViewerModel()

    # 3. Setup the animation loop
    q0 = robot.q0  # Get the neutral/default joint configuration
    dt = 0.01      # 100 Hz update rate
    t = 0.0

    print("Starting animation loop. Press Ctrl+C in the terminal to stop.")
    
    try:
        while True:
            # Create a copy of the default position
            q = q0.copy()
            
            # Create a smooth movement using sine and cosine waves
            # UR10 has 6 joints. Let's move the first three (pan, lift, elbow)
            q[0] += np.sin(t) * 1.0        # Shoulder Pan
            q[1] += np.sin(t * 0.5) * 0.5  # Shoulder Lift
            q[2] += np.cos(t * 1.2) * 1.2  # Elbow
            
            # Send the new joint configuration to the browser
            viz.display(q)
            
            # Advance time
            t += dt
            time.sleep(dt)
            
    except KeyboardInterrupt:
        print("\nAnimation stopped by user.")

if __name__ == '__main__':
    main()

Loading UR10 robot model...
Starting Meshcat server. Look for a new tab in your browser!
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/
Starting animation loop. Press Ctrl+C in the terminal to stop.
Opening in existing browser session.

Animation stopped by user.


In [36]:
q0 = np.array([0.0, -np.pi / 4, 0.0, -np.pi / 2, 0.0, np.pi / 3])
x0 = np.concatenate([q0, np.zeros(robot_pin_model.nv)])

# Create the cost functions
target = np.array([0.4, 0.0, 0.4])
target = np.array([0.6, 0.2, 0.0])
state = crocoddyl.StateMultibody(robot_pin_model)
frameTranslationResidual = crocoddyl.ResidualModelFrameTranslation(
    state, robot_pin_model.getFrameId("wrist_3_link"), target
)
goalTrackingCost = crocoddyl.CostModelResidual(state, frameTranslationResidual)
xRegCost = crocoddyl.CostModelResidual(state, crocoddyl.ResidualModelState(state))
uRegCost = crocoddyl.CostModelResidual(state, crocoddyl.ResidualModelControl(state))

# Create cost model per each action model
runningCostModel = crocoddyl.CostModelSum(state)
terminalCostModel = crocoddyl.CostModelSum(state)

# Then let's added the running and terminal cost functions
runningCostModel.addCost("gripperPose", goalTrackingCost, 1e2)
runningCostModel.addCost("stateReg", xRegCost, 1e-4)
runningCostModel.addCost("ctrlReg", uRegCost, 1e-7)
terminalCostModel.addCost("gripperPose", goalTrackingCost, 1e5)
terminalCostModel.addCost("stateReg", xRegCost, 1e-4)
terminalCostModel.addCost("ctrlReg", uRegCost, 1e-7)

# Running and terminal action models
DT = robot_mj_model.opt.timestep
actuationModel = crocoddyl.ActuationModelFull(state)
runningModel = crocoddyl.IntegratedActionModelEuler(
    crocoddyl.DifferentialActionModelFreeFwdDynamics(
        state, actuationModel, runningCostModel
    ),
    DT,
)
terminalModel = crocoddyl.IntegratedActionModelEuler(
    crocoddyl.DifferentialActionModelFreeFwdDynamics(
        state, actuationModel, terminalCostModel
    ),
    0.0,
)

In [45]:
# For this optimal control problem, we define 250 knots (or running action
# models) plus a terminal knot
T = 3000
print(len(x0[:6]))
print(x0)
u0 = np.zeros(actuationModel.nu * T)
robot_mj_data = mujoco.MjData(robot_mj_model)
print(robot_mj_data.xpos[mujoco.mj_name2id(robot_mj_model, mujoco.mjtObj.mjOBJ_BODY, "wrist_3_link")])
# compute xyz position of wrist_3_link using joint positions and forward kinematics
robot_data = robot_pin_model.createData()
pinocchio.forwardKinematics(robot_pin_model, robot_data, q0)
pinocchio.updateFramePlacements(robot_pin_model, robot_data)
frame_id = robot_pin_model.getFrameId("wrist_3_link")
wrist_pos = robot_data.oMf[frame_id].translation
print("wrist_3_link position (Pinocchio FK):", wrist_pos)

problem = crocoddyl.ShootingProblem(x0, [runningModel] * T, terminalModel)

# Creating the DDP solver for this OC problem, defining a logger
solver = crocoddyl.SolverFDDP(problem)
solver.solve()
log = crocoddyl.CallbackLogger()
pinocchio.forwardKinematics(robot_pin_model, robot_data, robot_mj_data.qpos)

print(robot_mj_data.ctrl)
with mujoco.viewer.launch_passive(robot_mj_model, robot_mj_data) as viewer:
    for u_cmd in solver.us:
        print("u_cmd:", u_cmd)
        robot_mj_data.ctrl[:] = u_cmd
        mujoco.mj_step(robot_mj_model, robot_mj_data)
        viewer.sync()
        time.sleep(robot_mj_model.opt.timestep)
    input("Press Enter to continue...")

# print position of wrist_3_link
print(robot_mj_data.xpos[mujoco.mj_name2id(robot_mj_model, mujoco.mjtObj.mjOBJ_BODY, "wrist_3_link")])

pinocchio.forwardKinematics(robot_pin_model, robot_data, robot_mj_data.qpos)
pinocchio.updateFramePlacements(robot_pin_model, robot_data)
frame_id = robot_pin_model.getFrameId("wrist_3_link")
wrist_pos = robot_data.oMf[frame_id].translation
print("wrist_3_link position (Pinocchio FK):", wrist_pos)

6
[ 0.         -0.78539816  0.         -1.57079633  0.          1.04719755
  0.          0.          0.          0.          0.          0.        ]
[0. 0. 0.]
wrist_3_link position (Pinocchio FK): [0.91923882 0.256141   1.04653882]


ArgumentError: Python argument types in
    SolverFDDP.solve(SolverFDDP, list)
did not match C++ signature:
    solve(crocoddyl::SolverFDDP {lvalue} self)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_xs)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_xs, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_us)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_xs, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_us, unsigned long maxiter)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1

In [53]:
import mujoco, mujoco.viewer
model = mujoco.MjModel.from_xml_path('../models/ur5e/ur5e.urdf')
data = mujoco.MjData(model)
with mujoco.viewer.launch_passive(model, data) as viewer:
    mujoco.mj_step(model, data)
    viewer.sync()
    input("Press Enter to continue...")
mujoco.mj_saveLastXML('../models/robot.xml', model)


ValueError: Error: Error opening file 'wrist1.stl'

In [55]:
from pathlib import Path
import mujoco, mujoco.viewer

urdf = (Path.cwd() / "models" / "ur5e" / "ur5e.urdf").resolve()
if not urdf.exists():
    urdf = (Path.cwd().parent / "models" / "ur5e" / "ur5e.urdf").resolve()

model = mujoco.MjModel.from_xml_path(str(urdf))
data = mujoco.MjData(model)

with mujoco.viewer.launch_passive(model, data) as viewer:
    mujoco.mj_step(model, data)
    viewer.sync()
    input("Press Enter to continue...")

mujoco.mj_saveLastXML(str(urdf.with_suffix(".xml")), model)


In [61]:
import mujoco
model = mujoco.MjModel.from_xml_path('../models/combined.xml')
data = mujoco.MjData(model)
with mujoco.viewer.launch_passive(model, data) as viewer:
    for _ in range(10):
        mujoco.mj_step(model, data)
        viewer.sync()
    input("Press Enter to continue...")


In [7]:
import pinocchio
import mujoco

# 1. Tworzenie MÓZGU (Crocoddyl / Pinocchio)
# Ładujemy czysty model ramienia w formacie URDF
urdf_path = "../models/ur5e/ur5e.urdf"
rmodel = pinocchio.buildModelFromUrdf(urdf_path)

# 2. Tworzenie ŚWIATA (MuJoCo)
# Opcja A: Jeśli symulujesz samo ramię, ładujesz ten sam URDF prosto do MuJoCo
mj_model = mujoco.MjModel.from_xml_path(urdf_path)

# 2. Zapisujemy go jako natywny plik MuJoCo (MJCF)
mujoco.mj_saveLastXML("../models/ur5e/ur5e.xml", mj_model)

print("Konwersja zakończona! Wygenerowano ur5e.xml")

# Opcja B: Jeśli masz złożoną scenę (podłoga, kamery, klocek, stół), 
# ładujesz plik scene.xml, który W SOBIE zawiera (include) plik URDF.
mj_model = mujoco.MjModel.from_xml_path("../models/test_scene.xml")

Konwersja zakończona! Wygenerowano ur5e.xml


ValueError: Error: Error opening file 'mesh/models/ur5e/collision/base.stl'